# Clustering Evaluation


Confronto tra run di clustering già **prodotte** (`fine_tuning: false`,
`results/lesion/clustering/production/<metodo>/<sessione>/`).

### Scopo

Leggere le metriche **dopo** la produzione. 

Due famiglie di metriche:

**1. Intrinseche** (sul singolo modello)
- Silhouette, Davies-Bouldin, Calinski-Harabasz 
- Dunn Index 
- AIC / BIC / Inerzia 

**2. Di confronto diretto** (tra 2+ run)
- ARI / NMI — `sklearn.metrics`, nessun codice nuovo
- Match matrix — `sklearn.metrics.cluster.contingency_matrix`, solo visualizzazione
- Subsampling Stability Index — nuovo, `consensus_clustering.subsampling_stability_index`: confronto singolo 100% vs 80% dei dati, non un ensemble di N ripetizioni come `run_monti_repeats`



In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

# Repo root isn't on sys.path when this notebook runs with cwd=notebooks/post-results_analysis/ -
# same fix as temp/*/run_prototype.py, needed here for the `from src...` imports below.
NOTEBOOK_REPO_ROOT = Path.cwd().resolve().parents[1]
if str(NOTEBOOK_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_REPO_ROOT))

from src.analysis.clustering_tuning import compute_clustering_metrics, dunn_index, run_clustering_tuning_sweep
from src.analysis.consensus_clustering import subsampling_stability_index
from src.utils.artifacts import load_matrix, read_run_params, save_matrix

sns.set_theme(style="whitegrid")


## 1. Dataset sintetico e run di clustering (stand-in)

4 blob non banalmente separati (stesso stile di `temp/kmeans/run_prototype.py`) — abbastanza
separati da avere una risposta "vera" riconoscibile, abbastanza vicini da rendere
interessanti i disaccordi tra metodi/k.

Vengono prodotte 4 run stand-in: `kmeans_k4` (k corretto), `kmeans_k6` (k sovra-specificato,
stesso metodo), `agglomerative_k4` (metodo diverso, stesso k) e `gmm_k4` (per avere anche
AIC/BIC in tabella) — scelte apposta per dare sia confronti "stesso metodo, k diverso" sia
"stesso k, metodo diverso".


In [ ]:
X, _true_labels = make_blobs(n_samples=300, centers=4, n_features=2, cluster_std=2.2, random_state=42)
subject_ids = [f"sub-SYNTH{i:04d}" for i in range(X.shape[0])]
print(f"dataset sintetico: {X.shape}")


In [ ]:
# Ogni entry: (nome_run, metodo CLUSTERING_METHODS, base_params, k/n_components da fissare).
# tuning_grid a un solo valore -> run_clustering_tuning_sweep fa esattamente 1 fit e calcola
# per noi tutte le metriche generiche + gli extra per-metodo (inertia/bic/aic) - stesso codice
# che gira in produzione/tuning, nessuna reimplementazione qui.
RUN_SPECS = [
    ("kmeans_k4", "kmeans", {"random_state": 0, "n_init": "auto"}, {"n_clusters": [4]}),
    ("kmeans_k6", "kmeans", {"random_state": 0, "n_init": "auto"}, {"n_clusters": [6]}),
    ("agglomerative_k4", "agglomerative", {"linkage": "ward"}, {"n_clusters": [4]}),
    ("gmm_k4", "gmm", {"random_state": 0}, {"n_components": [4]}),
]

OUTPUT_ROOT = NOTEBOOK_REPO_ROOT / "results/lesion/clustering/production"
SESSION_NAME = "notebook_synthetic_demo"  # nome esplicito, non ambiguo con una vera run


Per ogni configurazione: fit + metriche via `run_clustering_tuning_sweep`, poi le etichette
vengono scritte come colonna `cluster_label` di `metadata.csv` (stessa convenzione di
`clustering.py`, vedi `docs/dev/models.md` — "Where cluster labels live") tramite
`save_matrix`. `extra_metrics` (inertia/bic/aic) non fa parte del contratto ufficiale degli
artifact — viene comunque scritto in `config.md` (riga `Extra metrics: {...}`, convenzione
solo di questo notebook) così la tabella intrinseca sotto può rileggerlo dopo il round-trip
su disco, invece di tenerlo solo in memoria.


In [ ]:
# I 4 nomi generici che compute_clustering_metrics restituisce sempre - tutto il resto di una
# riga di run_clustering_tuning_sweep (inertia/bic/aic) e' un extra specifico del metodo.
GENERIC_METRIC_KEYS = {"silhouette", "calinski_harabasz", "davies_bouldin", "noise_fraction"}


def write_synthetic_run(name: str, method: str, base_params: dict, tuning_grid: dict) -> Path:
    results, labels_by_combo = run_clustering_tuning_sweep(method, X, base_params, tuning_grid)
    assert len(results) == 1, "tuning_grid a un solo valore deve produrre esattamente 1 riga"
    row = results.iloc[0].to_dict()
    (combo,) = labels_by_combo.keys()
    labels = labels_by_combo[combo]

    resolved_params = {**base_params, **dict(zip(tuning_grid.keys(), combo))}
    extra_metrics = {k: float(v) for k, v in row.items() if k not in tuning_grid and k not in GENERIC_METRIC_KEYS}

    metadata = pd.DataFrame({"subject_id": subject_ids, "cluster_label": labels})
    readme_lines = [
        f"# NEMESIS clustering (synthetic evaluation demo) — {method}",
        "",
        "Synthetic stand-in run (make_blobs) for notebooks/post-results_analysis/clustering_evaluation.ipynb",
        "(no real results/lesion/clustering production run existed yet - see docs/guides/evaluation.md).",
        "",
        f"Params used: {json.dumps(resolved_params)}",
        f"Extra metrics: {json.dumps(extra_metrics)}",
    ]

    output_dir = OUTPUT_ROOT / method / f"{SESSION_NAME}_{name}"
    save_matrix(output_dir, X, metadata, readme_lines, overwrite=True)
    return output_dir


run_dirs = {name: write_synthetic_run(name, method, base_params, tuning_grid) for name, method, base_params, tuning_grid in RUN_SPECS}
run_dirs


### Caricamento tramite il contratto reale degli artifact

Da qui in poi ogni run viene letta **solo** da disco con `load_matrix(input_dir)` — lo stesso
loader di `clustering.py`/`embedding_app.py` — e i suoi parametri risolti con
`read_run_params(input_dir)` (stesso helper usato da `embedding_app.py`/`clustering.py`'s
`viz_embedding_path` cross-check). `extra_metrics` viene riletto dalla riga `Extra metrics:`
del `config.md` con un piccolo parser locale (convenzione di questo notebook, non del
contratto ufficiale di `artifacts.py`).


In [ ]:
def read_extra_metrics(run_dir: Path) -> dict:
    """Notebook-only convention (not part of artifacts.py's contract): the 'Extra metrics: {...}'
    line write_synthetic_run adds to config.md alongside the real 'Params used:' line."""
    for line in (run_dir / "config.md").read_text().splitlines():
        if line.startswith("Extra metrics: "):
            return json.loads(line[len("Extra metrics: ") :])
    raise ValueError(f"{run_dir}/config.md has no 'Extra metrics:' line")


runs = {}
for name, input_dir in run_dirs.items():
    run_X, run_metadata, _extra_arrays = load_matrix(input_dir)
    method = input_dir.parent.name  # OUTPUT_ROOT/<method>/<session_name>/, see write_synthetic_run
    runs[name] = {
        "input_dir": input_dir,
        "method": method,
        "X": run_X,
        "metadata": run_metadata,
        "labels": run_metadata["cluster_label"].to_numpy(),
        "params": read_run_params(input_dir),
        "extra_metrics": read_extra_metrics(input_dir),
    }

for name, run in runs.items():
    print(f"{name} ({run['method']}): {run['X'].shape[0]} soggetti, {len(np.unique(run['labels']))} cluster, params={run['params']}")


## 2. Metriche intrinseche

Una riga per run: Silhouette / Calinski-Harabasz / Davies-Bouldin / noise_fraction (riusa
`compute_clustering_metrics`, non reimplementata), Dunn Index (nuovo), più `inertia`
(kmeans) o `bic`/`aic` (gmm) quando presenti — nessuna riga li ha entrambi, `NaN` altrove.


In [ ]:
intrinsic_rows = []
for name, run in runs.items():
    metrics = compute_clustering_metrics(run["X"], run["labels"])
    metrics["dunn"] = dunn_index(run["X"], run["labels"])
    metrics.update(run["extra_metrics"])
    intrinsic_rows.append({"run": name, **metrics})

intrinsic_table = pd.DataFrame(intrinsic_rows).set_index("run")
intrinsic_table


**Lettura** (nessuna scelta automatica, solo osservazioni): `kmeans_k6` è atteso peggiore di
`kmeans_k4` su Silhouette/Dunn/Davies-Bouldin (k sovra-specificato su 4 blob veri) — se non lo
fosse, andrebbe capito perché prima di fidarsi delle altre run. `gmm_k4` porta AIC/BIC, non
direttamente confrontabili con Silhouette/Dunn/CH/DB (scale diverse) — servono a confrontare
GMM contro *altri* `n_components`/`covariance_type` di GMM, non contro run di k-means/gerarchico.


## 3. Metriche di confronto diretto: ARI / NMI

Per ogni coppia di run: Adjusted Rand Index e Normalized Mutual Information
(`sklearn.metrics`, nessun calcolo nuovo). Le run condividono lo stesso `X`/ordine dei
soggetti per costruzione (stesso dataset sintetico) — l'allineamento per `subject_id` viene
comunque verificato esplicitamente prima del confronto, non assunto silenziosamente (una vera
run di produzione potrebbe avere un sottoinsieme di soggetti diverso).


In [ ]:
def aligned_labels(name_a: str, name_b: str) -> tuple[np.ndarray, np.ndarray]:
    subjects_a = runs[name_a]["metadata"]["subject_id"]
    subjects_b = runs[name_b]["metadata"]["subject_id"]
    if not subjects_a.equals(subjects_b):
        raise ValueError(
            f"{name_a!r} and {name_b!r} don't share the exact same subject_id order - align "
            "them explicitly (e.g. a merge on subject_id) before comparing labels positionally"
        )
    return runs[name_a]["labels"], runs[name_b]["labels"]


run_names = list(runs.keys())
ari_table = pd.DataFrame(index=run_names, columns=run_names, dtype=float)
nmi_table = pd.DataFrame(index=run_names, columns=run_names, dtype=float)
for name_a in run_names:
    for name_b in run_names:
        labels_a, labels_b = aligned_labels(name_a, name_b)
        ari_table.loc[name_a, name_b] = adjusted_rand_score(labels_a, labels_b)
        nmi_table.loc[name_a, name_b] = normalized_mutual_info_score(labels_a, labels_b)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(ari_table, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="viridis", ax=axes[0])
axes[0].set_title("Adjusted Rand Index")
sns.heatmap(nmi_table, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="viridis", ax=axes[1])
axes[1].set_title("Normalized Mutual Information")
fig.tight_layout()


## 4. Match matrix (contingency) tra una coppia di run

`kmeans_k4` vs `agglomerative_k4` — stesso `k`, metodo diverso: la coppia più interessante da
ispezionare a occhio (un disaccordo qui non è spiegabile da un semplice sovra/sotto-specificare
`k`, va letto come un vero disaccordo tra criteri di raggruppamento).


In [ ]:
name_a, name_b = "kmeans_k4", "agglomerative_k4"
labels_a, labels_b = aligned_labels(name_a, name_b)
match_matrix = contingency_matrix(labels_a, labels_b)
match_df = pd.DataFrame(
    match_matrix,
    index=[f"{name_a}={c}" for c in sorted(np.unique(labels_a))],
    columns=[f"{name_b}={c}" for c in sorted(np.unique(labels_b))],
)

plt.figure(figsize=(7, 6))
sns.heatmap(match_df, annot=True, fmt="d", cmap="Blues")
plt.title(f"Match matrix: {name_a} vs {name_b}")
plt.yticks(rotation=0)
plt.tight_layout()
match_df


## 5. Subsampling Stability Index

Per `kmeans_k4` e `agglomerative_k4`: rifit dello stesso metodo/parametri (letti da
`config.md` via `read_run_params`, non ridichiarati a mano) su un sottocampione all'80% dei
soggetti, poi ARI tra il fit sul 100% e quello sull'80% (`subsampling_stability_index`,
nuovo — confronto singolo, non un ensemble di N ripetizioni come `run_monti_repeats`, vedi
il suo docstring in `consensus_clustering.py` per la differenza).


In [ ]:
stability_rows = []
for name in ["kmeans_k4", "agglomerative_k4"]:
    run = runs[name]
    index = subsampling_stability_index(run["method"], run["X"], run["params"], subsample_fraction=0.8)
    stability_rows.append({"run": name, "subsampling_stability_index": index})

pd.DataFrame(stability_rows).set_index("run")


**Lettura**: un indice vicino a 1 significa che il clustering non dipende da un piccolo
sottoinsieme di soggetti (es. outlier) — un valore basso è un segnale da approfondire prima
di fidarsi della run, non un criterio per scartarla automaticamente.


## Riepilogo

- Le metriche **intrinseche** (§2) si leggono solo *dentro* la stessa run/metodo — non hanno
  senso per confrontare `kmeans` con `gmm` (scale diverse, es. AIC/BIC).
- ARI/NMI (§3) e la match matrix (§4) sono gli strumenti giusti per confrontare run/metodi
  diversi tra loro.
- Il Subsampling Stability Index (§5) risponde a una domanda diversa: *quella singola run* è
  robusta al campione esatto di soggetti, o è un artefatto?
- Nessuna cella sopra rifà la scelta di `k`/metodo — quella è già stata fatta in tuning,
  prima che queste run venissero prodotte; qui si valuta a posteriori come sono andate
  (`code_standards.md` §0: nessuna selezione automatica, né in tuning né qui).

Guida completa: `docs/guides/evaluation.md`.
